# MiniTrain validation notebook
Notebook-first smoke validation for configs, LR scheduling, Dense/MoE training, checkpoint restore, and the optional CUDA/Triton path.

In [ ]:
from pathlib import Path
import sys
import tempfile
import torch

ROOT = Path.cwd()
if ROOT.name == 'tests':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from minitrain.kernels.torch_ops import TorchOpsBackend
from minitrain.kernels.triton.router import is_router_postprocess_supported
from minitrain.model import MiniTransformer, ModelConfig
from minitrain.runtime.config import LRSchedulerConfig, load_yaml_dict
from minitrain.train.checkpoint import restore_training_checkpoint, save_checkpoint
from minitrain.train.lr_scheduler import LearningRateScheduler
from minitrain.train.precision import resolve_precision_policy

In [ ]:
# Parse every shipped model preset.
model_paths = sorted((ROOT / 'configs').glob('model_*.yaml'))
model_configs = [ModelConfig(**load_yaml_dict(path)['model']) for path in model_paths]
[(path.name, cfg.ffn_type, cfg.seq_len) for path, cfg in zip(model_paths, model_configs)]

In [ ]:
# Constant/warmup/cosine LR behavior.
parameter = torch.nn.Parameter(torch.ones(()))
optimizer = torch.optim.AdamW([parameter], lr=3e-4)
scheduler = LearningRateScheduler(
    optimizer,
    LRSchedulerConfig(schedule='cosine', warmup_steps=2, min_lr_ratio=0.1),
    total_steps=10,
)
curve = []
for step in range(11):
    scheduler.step(step)
    curve.append(scheduler.get_last_lr()[0])
assert curve[1] == 3e-4 and abs(curve[-1] - 3e-5) < 1e-12
curve

In [ ]:
# Dense and MoE forward/backward on CPU.
def tiny_config(ffn_type):
    return ModelConfig(
        vocab_size=97, seq_len=8, n_layers=2, n_heads=4, hidden_size=32,
        intermediate_size=24, ffn_type=ffn_type, num_experts=4, experts_per_token=2,
    )

models = {}
for ffn_type in ('dense', 'moe'):
    cfg = tiny_config(ffn_type)
    model = MiniTransformer(cfg, TorchOpsBackend())
    tokens = torch.randint(cfg.vocab_size, (2, cfg.seq_len))
    loss, logits = model(tokens, targets=tokens)
    loss.backward()
    assert torch.isfinite(loss) and logits.shape == (2, cfg.seq_len, cfg.vocab_size)
    models[ffn_type] = model

required_moe_metrics = {
    'moe/router_entropy', 'moe/dropped_route_fraction',
    'moe/max_expert_load', 'moe/min_expert_load',
    'moe/aux_loss', 'moe/z_loss',
}
assert required_moe_metrics <= models['moe'].last_moe_metrics.keys()
assert all(torch.isfinite(value) for value in models['moe'].last_moe_metrics.values())
assert all(
    block.ffn.router.weight.grad is not None
    and torch.isfinite(block.ffn.router.weight.grad).all()
    for block in models['moe'].blocks
)
models['moe'].last_moe_metrics

In [ ]:
# Full checkpoint save/restore, including scheduler state.
model = models['dense']
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = LearningRateScheduler(optimizer, LRSchedulerConfig(warmup_steps=0), total_steps=10)
scheduler.step(4)
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'smoke.pt'
    save_checkpoint(path, model, optimizer, 4, epoch=1, tokens_seen=128, lr_scheduler=scheduler)
    restored = restore_training_checkpoint(
        path, model, optimizer, lr_scheduler=scheduler, restore_rng=True
    )
assert restored == {'step': 4, 'epoch': 1, 'tokens_seen': 128}
restored

In [ ]:
# Optional CUDA/Triton MoE training step.
if torch.cuda.is_available():
    from minitrain.kernels.triton import TritonOpsBackend
    cfg = ModelConfig(
        vocab_size=97, seq_len=8, n_layers=1, n_heads=4, hidden_size=64,
        intermediate_size=32, ffn_type='moe', num_experts=4, experts_per_token=2,
    )
    precision = resolve_precision_policy('auto', torch.device('cuda'))
    model = MiniTransformer(
        cfg, TritonOpsBackend(), activation_dtype=precision.activation_dtype
    ).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda', enabled=precision.grad_scaling_enabled)
    tokens = torch.randint(cfg.vocab_size, (2, cfg.seq_len), device='cuda')
    with precision.autocast_context(torch.device('cuda')):
        loss, _ = model(tokens, targets=tokens, use_fused_loss=True)
    scaler.scale(loss).backward()
    router = model.blocks[0].ffn.router
    experts = model.blocks[0].ffn.gate_up_proj
    assert torch.isfinite(loss)
    assert router.weight.grad is not None and torch.isfinite(router.weight.grad).all()
    assert experts.grad is not None and torch.isfinite(experts.grad).all()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    scaler.step(optimizer)
    scaler.update()
    assert required_moe_metrics <= model.last_moe_metrics.keys()
    probe = torch.randn(2, cfg.num_experts, device='cuda', dtype=torch.float32)
    native_router = is_router_postprocess_supported(
        probe, cfg.experts_per_token, cfg.router_normalize_topk
    )
    print(
        'CUDA/Triton MoE smoke passed:', float(loss),
        'precision:', precision.name, 'loss scaling:', scaler.is_enabled(),
        'native router:', native_router,
    )
else:
    print('CUDA unavailable; optional Triton cell skipped.')